# BirdCLEF 2026 - Phase 1 Improved Inference

## Overview
This notebook loads Phase 1 improved models and generates predictions for test soundscapes.

**Phase 1 Model Features:**
- tf_efficientnet_b0_ns backbone (better pretrained weights)
- Higher resolution mel-spectrograms (n_mels=224)
- Trained on train_audio + train_soundscapes
- SpecAugment and Mixup augmentation
- 20 epochs with LR warmup

**Inference Features:**
- Model ensemble (average predictions from multiple folds)
- Processes 1-minute soundscapes into 5-second segments
- Optimized for CPU runtime

**Competition:** https://www.kaggle.com/competitions/birdclef-2026

## 1. Setup and Imports

In [ ]:
import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Audio processing
import librosa
import soundfile as sf

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm

warnings.filterwarnings('ignore')

print(f"timm version: {timm.__version__}")
print(f"librosa version: {librosa.__version__}")
print("✅ All packages available")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Configuration - PHASE 1 IMPROVED

In [ ]:
class CFG:
    # Paths
    data_dir = Path('/kaggle/input/birdclef-2026')
    test_soundscapes_dir = data_dir / 'test_soundscapes'
    model_dir = Path('/kaggle/input/birdclef-2026-phase1-models')  # Upload Phase 1 trained models here
    output_dir = Path('/kaggle/working')
    
    # Audio parameters - PHASE 1 IMPROVED (must match training)
    sample_rate = 32000
    duration = 5  # seconds
    n_mels = 224  # CHANGED from 128 - higher resolution
    fmin = 20
    fmax = 16000
    n_fft = 2048
    hop_length = 512
    
    # Model parameters - PHASE 1 IMPROVED
    model_name = 'tf_efficientnet_b0_ns'  # CHANGED - better pretrained weights
    num_classes = 234
    
    # Inference parameters
    batch_size = 16
    num_workers = 0
    device = 'cpu'  # CPU only for submission
    
    # Model ensemble
    model_folds = [0]  # Which folds to use for ensemble (add more as you train)
    
CFG.output_dir.mkdir(exist_ok=True, parents=True)
print(f"Device: {CFG.device}")
print(f"\n🚀 PHASE 1 INFERENCE CONFIGURATION:")
print(f"  ✅ Model: {CFG.model_name}")
print(f"  ✅ Mel bins: {CFG.n_mels}")
print(f"  ✅ Ensemble folds: {CFG.model_folds}")

## 3. Load Metadata

In [ ]:
# Load sample submission
sample_submission = pd.read_csv(CFG.data_dir / 'sample_submission.csv')
print(f"Submission shape: {sample_submission.shape}")
print(f"Number of test segments: {len(sample_submission)}")

# Get species columns
species_columns = [col for col in sample_submission.columns if col != 'row_id']
print(f"Number of species: {len(species_columns)}")

sample_submission.head()

In [ ]:
# Load taxonomy for reference
taxonomy_df = pd.read_csv(CFG.data_dir / 'taxonomy.csv')
species_list = taxonomy_df['primary_label'].tolist()
species_to_idx = {species: idx for idx, species in enumerate(species_list)}

print(f"Total species in taxonomy: {len(species_list)}")

## 4. Audio Processing Functions

In [ ]:
def load_audio_segment(filepath, start_time, duration=5, sr=32000):
    """Load a specific segment from an audio file"""
    try:
        # Calculate offset in samples
        offset = int(start_time * sr)
        num_samples = int(duration * sr)
        
        # Load audio segment
        audio, orig_sr = sf.read(filepath, start=offset, frames=num_samples)
        
        # Resample if needed
        if orig_sr != sr:
            audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=sr)
        
        # Convert to mono if stereo
        if len(audio.shape) > 1:
            audio = audio.mean(axis=1)
        
        # Ensure correct length
        target_length = sr * duration
        if len(audio) < target_length:
            audio = np.pad(audio, (0, target_length - len(audio)), mode='constant')
        elif len(audio) > target_length:
            audio = audio[:target_length]
        
        return audio.astype(np.float32)
    
    except Exception as e:
        print(f"Error loading {filepath} at {start_time}s: {e}")
        return np.zeros(sr * duration, dtype=np.float32)


def audio_to_melspectrogram(audio, sr=32000, n_mels=128, fmin=20, fmax=16000, n_fft=2048, hop_length=512):
    """Convert audio to mel-spectrogram"""
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax,
        n_fft=n_fft,
        hop_length=hop_length
    )
    
    # Convert to dB scale
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalize to [0, 1]
    mel_spec_norm = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)
    
    return mel_spec_norm.astype(np.float32)

## 5. Create Test Dataset

In [ ]:
def parse_row_id(row_id):
    """Parse row_id to get filename and end_time"""
    # Format: BC2026_Test_0001_S05_20250227_010002_20
    parts = row_id.rsplit('_', 1)
    filename = parts[0] + '.ogg'
    end_time = int(parts[1])
    start_time = end_time - 5
    return filename, start_time

# Parse all row IDs
test_data = []
for row_id in sample_submission['row_id']:
    filename, start_time = parse_row_id(row_id)
    test_data.append({
        'row_id': row_id,
        'filename': filename,
        'start_time': start_time
    })

test_df = pd.DataFrame(test_data)
print(f"Test segments: {len(test_df)}")
print(f"Unique soundscapes: {test_df['filename'].nunique()}")
test_df.head()

In [ ]:
class TestDataset(Dataset):
    def __init__(self, df, audio_dir, cfg):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.cfg = cfg
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load audio segment
        filepath = self.audio_dir / row['filename']
        audio = load_audio_segment(
            filepath,
            row['start_time'],
            duration=self.cfg.duration,
            sr=self.cfg.sample_rate
        )
        
        # Convert to mel-spectrogram
        mel_spec = audio_to_melspectrogram(
            audio,
            sr=self.cfg.sample_rate,
            n_mels=self.cfg.n_mels,
            fmin=self.cfg.fmin,
            fmax=self.cfg.fmax,
            n_fft=self.cfg.n_fft,
            hop_length=self.cfg.hop_length
        )
        
        # Convert to 3-channel image
        mel_spec = np.stack([mel_spec, mel_spec, mel_spec], axis=0)
        
        return {
            'image': torch.tensor(mel_spec, dtype=torch.float32),
            'row_id': row['row_id']
        }

## 6. Model Definition - PHASE 1 IMPROVED

In [ ]:
class BirdCLEFModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=234, pretrained=False):
        super().__init__()
        
        # Load backbone
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool=''
        )
        
        # Get number of features - PHASE 1 UPDATED for higher resolution
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 224, 313)  # Updated for n_mels=224
            features = self.backbone(dummy_input)
            n_features = features.shape[1]
        
        # Global pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_features, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        pooled = self.global_pool(features)
        pooled = pooled.view(pooled.size(0), -1)
        output = self.classifier(pooled)
        return output

## 7. Load Models

In [ ]:
%%time
# Load trained models
models = []

for fold in CFG.model_folds:
    model = BirdCLEFModel(CFG.model_name, CFG.num_classes, pretrained=False)
    
    # Load weights
    model_path = CFG.model_dir / f'best_model_fold{fold}.pth'
    
    if model_path.exists():
        model.load_state_dict(torch.load(model_path, map_location=CFG.device))
        model = model.to(CFG.device)
        model.eval()
        models.append(model)
        print(f"✓ Loaded Phase 1 model for fold {fold}")
    else:
        print(f"✗ Model not found for fold {fold}: {model_path}")

print(f"\nTotal models loaded: {len(models)}")

## 8. Generate Predictions

In [ ]:
def predict(models, dataloader, device):
    """Generate predictions using model ensemble"""
    all_preds = []
    all_row_ids = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting"):
            images = batch['image'].to(device)
            row_ids = batch['row_id']
            
            # Ensemble predictions
            batch_preds = []
            for model in models:
                outputs = model(images)
                preds = torch.sigmoid(outputs).cpu().numpy()
                batch_preds.append(preds)
            
            # Average predictions
            avg_preds = np.mean(batch_preds, axis=0)
            
            all_preds.append(avg_preds)
            all_row_ids.extend(row_ids)
    
    # Concatenate all predictions
    all_preds = np.concatenate(all_preds, axis=0)
    
    return all_preds, all_row_ids

In [ ]:
%%time
# Create test dataset and dataloader
test_dataset = TestDataset(test_df, CFG.test_soundscapes_dir, CFG)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=False
)

print(f"Test dataset size: {len(test_dataset)}")
print(f"Number of batches: {len(test_loader)}")

In [ ]:
%%time
# Generate predictions
predictions, row_ids = predict(models, test_loader, CFG.device)

print(f"\nPredictions shape: {predictions.shape}")
print(f"Number of row IDs: {len(row_ids)}")

## 9. Create Submission File

In [ ]:
# Create submission dataframe
submission = pd.DataFrame({
    'row_id': row_ids
})

# Add predictions for each species
for idx, species in enumerate(species_columns):
    submission[species] = predictions[:, idx]

# Verify submission format
print(f"Submission shape: {submission.shape}")
print(f"Expected shape: {sample_submission.shape}")
print(f"\nColumns match: {list(submission.columns) == list(sample_submission.columns)}")

submission.head()

In [ ]:
# Check prediction statistics
print("Prediction Statistics:")
print(f"Min: {predictions.min():.6f}")
print(f"Max: {predictions.max():.6f}")
print(f"Mean: {predictions.mean():.6f}")
print(f"Median: {np.median(predictions):.6f}")
print(f"\nPredictions > 0.5: {(predictions > 0.5).sum()} / {predictions.size}")
print(f"Predictions > 0.1: {(predictions > 0.1).sum()} / {predictions.size}")

In [ ]:
# Save submission
submission.to_csv('submission.csv', index=False)
print("✓ Submission file saved!")

# Verify file size
file_size = os.path.getsize('submission.csv') / (1024 * 1024)
print(f"File size: {file_size:.2f} MB")

## 10. Validation

In [ ]:
# Check for any issues
print("Submission Validation:")
print(f"✓ Shape: {submission.shape}")
print(f"✓ Columns: {len(submission.columns)}")
print(f"✓ Rows: {len(submission)}")
print(f"✓ No NaN values: {not submission.isnull().any().any()}")
print(f"✓ All values in [0, 1]: {(submission[species_columns] >= 0).all().all() and (submission[species_columns] <= 1).all().all()}")
print(f"✓ Row IDs match: {(submission['row_id'] == sample_submission['row_id']).all()}")

## Summary - PHASE 1 INFERENCE

This Phase 1 improved inference notebook:
- ✅ Loads Phase 1 trained models (tf_efficientnet_b0_ns)
- ✅ Uses higher resolution mel-spectrograms (n_mels=224)
- ✅ Processes test soundscapes into 5-second segments
- ✅ Generates predictions using model ensemble
- ✅ Creates valid submission file
- ✅ Optimized for CPU runtime

**Phase 1 Improvements:**
- Better pretrained model (tf_efficientnet_b0_ns)
- Higher resolution features (224 mel bins vs 128)
- Trained with SpecAugment and Mixup
- Trained on train_audio + train_soundscapes
- Extended training (20 epochs)

**Expected Improvement:** +0.03-0.05 over baseline (0.80 → 0.83-0.85)

**Submission File:** `submission.csv`

**Next Steps:**
1. Submit to competition
2. Compare with baseline score (0.802)
3. If successful, train remaining folds and ensemble
4. Move to Phase 2 improvements

**Expected Runtime:** ~30-60 minutes on CPU